In [ ]:
#1----------Imports & Setup
import string         # to get a list of punctuation
import collections    # to calculate frequency
'''The collections module in Python provides alternative data structures
to Python's built-in containers (like lists, dictionaries, etc.). This line imports the defaultdict class from the collections module'''
from collections import defaultdict

#2-----------Text Input
#first we define a function to get the text to analyze
def get_user_input():
    print("--- Welcome to The Intelligent Text Analyzer! ---")
    print("--- Please choose a Way to insert your text  ---")
    print("1. Direct Text Entry (d)")
    print("2. File Path Input (f)")

    choice = input("Select (d/f): ").lower()

    if choice == 'f':
        # the choice of inserting the text by file
        file_path = input("Please enter the full path to your text file: ")
        try:
            with open(file_path, 'r', encoding='utf-8') as file:
                return file.read()
        except FileNotFoundError:
            print("Error: File not found!")    # error handling in case the the file was not found
            return ""

    elif choice == 'd':
        # the choice of inserting the text directly
        print("Enter your text. Type '$$END TEXT$$' on a new line to finish:")
        lines = []
        while True:
            line = input()
            if line == "$$END TEXT$$":
                break
            lines.append(line)
        return " ".join(lines)

    return ""



#3---------- pre-process and prepare the text
# the goal: convert the text to lowercase and remove punctuation to standardize the data
def preprocess_text(text):
    # convert to lowercase
    text = text.lower()

    # getting the sentences before removing the dots
    sentences= text.split(".")
    sentences= [s.strip() for s in sentences if s.strip()]


    # remove punctuation like (?,!,.)
    # we use string.punctuation to get a list of all the punctuation
    translator = str.maketrans('', '', string.punctuation)
    clean_text = text.translate(translator)

    # turning the text into a list of words for easier processing
    words = clean_text.split()
    return clean_text, words, sentences


# smart Autocomplete (Trie)


class TrieNode:
    '''This class represents a single node in the Trie Tree. Its purpose is to manage the connections to its child nodes
    and indicate whether it marks the end of a word.'''
    '''__init__(self) initializes a new TrieNode object. Child nodes are represented by an empty dictionary {}
     and the Boolean value is_word indicates whether it's the end of a word (default is False).'''
    def __init__(self):
        self.children= defaultdict(TrieNode) #defaultdict can be used to initialize child nodes automatically when traversing or inserting nodes.
        self.is_word= False
        self.frequency= 0

class Trie:
    '''This class manages the overall Trie Tree and provides methods for inserting and searching for words.'''
    def __init__(self):
        self.root= TrieNode() #__init__(self) creates a new TrieTree object and initializes its root node.

    # Add a word to the Trie, Time Complexity: O(n) where n is the word length
    '''insert(self, word: str) accepts a word as an argument and adds it to the Trie Tree. Starting from the root node,
     it iterates through the letters of the word creating new child nodes if necessary and sets the final node's is_word attribute to True.'''
    def insert(self, word):
        node= self.root
        for char in word:
            node= node.children[char]
        node.is_word= True
        node.frequency += 1

    # Search for all words that start with the given prefix
    # Returns top 5 suggestions sorted by frequency
    '''search(self, prefix: str) -> bool returns a boolean value indicating whether a specified prefix exists in the Trie Tree.
    Iterating through the prefix's characters, it looks for corresponding child nodes until
    finding the last character of the prefix or reaching a dead end. At the end of the loop,
    if the last node's is_word attribute is True, the method returns True and add the suggest prefix in suggestions list.
    Otherwise, it returns False.'''
    def search(self, prefix):
        node= self.root
        suggestions= [] #store the words that match the prefix.

        for char in prefix:
            if char not in node.children: #checks if there is a child node corresponding to that character in the current node’s children dictionary.
                return suggestions #returns the list of suggestions that match the given prefix
            node = node.children[char]

        self._find_suggestions(node, prefix, suggestions)#calls _find_suggestions with the current node, the prefix, and the suggestions list.
        '''This helper method recursively traverses all child nodes from the current node, collecting all complete words that start
        with the given prefix and adding them to the suggestions list.'''
        suggestions.sort(key=lambda x: x[1], reverse=True)

        return [word for word, freq in suggestions[:5]]

    # Recursive traversal to collect all words under a given prefix node
    def _find_suggestions(self, node, prefix, suggestions):
        if node.is_word:
            suggestions.append((prefix, node.frequency))
        for char, child in node.children.items():
            self._find_suggestions(child, prefix + char, suggestions)

#4--------N-grams
#contextual Next Word Prediction
# Maps each word to the words that frequently follow it
# Uses nested defaultdict for automatic frequency counting
def build_bigram_counts(words):
    bigram_counts= defaultdict(lambda: defaultdict(int))
    for i in range(len(words)-1):
        prev_word= words[i]
        next_word= words[i+1]
        bigram_counts[prev_word][next_word] += 1 #counts occurrences of second word w2 after w1
    return bigram_counts
# Build Trigram model:
# Maps word pairs to possible next words for better contextual prediction
def build_trigram_counts(words):
    trigram_counts= defaultdict(lambda: defaultdict(int))
    for i in range(len(words)-2):
        prev_words= (words[i], words[i+1])
        next_word= words[i+2]
        trigram_counts[prev_words][next_word] += 1 #counts occurrences of third word w3 after (w1, w2)
    return trigram_counts
# Predict next word based on context using trigram first, then bigram fallback
def predict_next_word(input_words, bigram_counts, trigram_counts, top_n=5):
    words= input_words.lower().split()

    if len(words) >= 2:
        context= tuple(words[-2:])
        if context in trigram_counts:
            next_words= trigram_counts[context]
            return sorted(next_words, key=next_words.get, reverse=True)[:top_n]

    if len(words) >= 1:
        context= words[-1]
        if context in bigram_counts:
            next_words= bigram_counts[context]
            return sorted(next_words, key=next_words.get, reverse=True)[:top_n]

    return []


#5-----------starting to run the program and receive the text
raw_data= get_user_input()
clean_text, words_list, sentences_list= preprocess_text(raw_data) #str,list,list data strectuer

if not words_list:
    print("\n[Warning] No text loaded. Please restart and enter some text.")
else:
    print(f"\n[Success] Loaded {len(words_list)} words.")


    #building Core Data Structures after the preprocessing
    # we defined these here to be globally accessible by all intelligent features in the system


    word_counts= collections.Counter(words_list) # Counter used for efficient word frequency computation (O(1) average lookup)
    unique_words= set(words_list) # Set used to store unique vocabulary for fast membership checking

    bigram_counts= build_bigram_counts(words_list)
    trigram_counts= build_trigram_counts(words_list)

    #stop words set
    # Set of common stop words excluded from keyword extraction
    # Using a set allows O(1) lookup time
    stop_words= {
        "a", "all", "also", "am", "an", "and", "any", "are", "as", "at",
        "be", "because", "been", "both", "but", "by", "can", "could",
        "did", "do", "does", "doing", "each", "few", "for", "from",
        "had", "has", "have", "having", "he", "her", "here", "hers",
        "herself", "him", "himself", "his", "how", "however", "i", "if",
        "in", "is", "it", "its", "itself", "just", "me", "more", "most",
        "my", "myself",  "nor",  "now", "of", "off", "on",
        "once", "only", "or", "other", "our", "ours", "ourselves", "out",
        "own", "s", "shall", "she", "should", "so", "some", "such", "t",
        "than", "that", "the", "their", "theirs", "them", "themselves",
        "then", "there", "these", "they", "this", "those", "through",
        "to", "too", "under", "until", "up", "very", "was", "were",
        "what", "whatever", "when", "where", "whether", "which", "while",
        "who", "whom", "why", "will", "with", "would", "you", "your",
        "yours", "yourself", "yourselves"

    }

    #sentiment lexicons
    positive_words= {
        "good", "great", "best", "better", "happy","happiness" ,"yes", "true",
        "right", "correct", "well", "nice", "awesome", "excellent",
        "positive", "safe", "love", "like", "glad", "fine",
        "amazing", "beautiful", "bright", "celebrate", "clean",
        "comfortable", "easy", "enjoy", "fair", "fantastic",
        "favorite", "friendly", "fun", "generous", "helpful",
        "honest", "ideal", "important", "kind", "lucky",
        "perfect", "pleasant", "proud", "quality", "recommend",
        "satisfied", "smile", "strong", "success", "super",
        "thanks", "useful", "value", "win", "wonderful", "worth", "optimistic",
        "kind","joyous","cheerful", "cute", " delightful","priceless"

    }

    negative_words= {
        "bad", "worse", "worst",  "none", "untrue","lie","cheat",
        "wrong", "false", "fail", "error", "hard", "poor",
        "sad", "hate", "stop", "against", "low", "negative",
        "annoy", "awful", "boring", "broken", "cheap", "complain",
        "confused", "danger", "dead", "defect", "difficult",
        "dirty", "disappoint", "doubt", "dreadful", "evil",
        "fault", "fear", "garbage", "guilt", "harm", "hate",
        "horrible", "hurt", "ignore", "impossible", "junk",
        "lack", "loss", "mistake", "pain", "problem", "refuse",
        "reject", "scary", "sick", "slow", "stupid", "terrible",
        "ugly", "unfair", "unhappy", "useless", "waste", "weak", "worry","worrying",
        "pale","dul","kill","damage","wicked","colorless","worthless","death","die"

    }

    # negation words so we can handle cases like "not good"
    negation_words= {
        "not", "no", "never", "none", "neither", "nor", "nothing",
        "nowhere", "hardly", "scarcely", "barely", "doesn't",
        "don't", "didn't", "isn't", "aren't", "wasn't", "weren't",
        "hasn't", "haven't", "hadn't", "won't", "wouldn't",
        "can't", "cannot", "couldn't", "shouldn't", "mightn't",
        "mustn't", "without", "against", "lack", "lacking",'un'
    }

    #building Trie for Autocomplete
    trie= Trie()
    for word in words_list:
        trie.insert(word)
    undo_stack = []
    # --- Interactive Menu System ---
    # Allows the user to continuously select different operations until exit
    while True:
        print("\n" + "="*35)
        print("--- SMART TEXT ANALYZER MENU ---")
        print("1. Word Statistics (Total & Unique)")
        print("2. Search for a Word/Phrase")
        print("3. Word Replacement")
        print("4. Character Statistics")
        print("5. Keyword Extraction")
        print("6. Sentiment Analysis")
        print("7. Smart Autocomplete")
        print("8. Next Word Prediction")
        print("9. Undo Last Replacement")
        print("10. Exit Program")
        print("="*35)

        choice = input("Select an option (1-10): ")

        # choice 1: Word Statistics
        # Feature 1: Display total words, unique words, and top frequent words
        if choice == '1':
            total_words = len(words_list)
            unique_words = set(words_list)

            print(f"\nTotal Number of Words: {total_words}")
            print(f"Number of Unique Words: {len(unique_words)}")
            '''The Counter class was used to efficiently count the frequency of each word in the text,
             providing O(1) average lookup time and easy extraction of the most common words.'''
            counts = collections.Counter(words_list)
            print("Top 5 most frequent words:", counts.most_common(5))

        # choice 2: Search
        # Feature 2: Search for a word and report its position (sentence number and word index)
        elif choice == '2':
            search_term = input("Enter the word to search for: ").lower()
            found= False
            #The enumerate() function was used to iterate through lists while keeping track of element positions efficiently and clearly.
            for sentence_index, sentence in enumerate(sentences_list):
                sentence_clean = sentence.translate(str.maketrans('', '', string.punctuation))
                sentence_words= sentence_clean.split()
                for word_index, word in enumerate(sentence_words):
                    if word == search_term:
                        print(f"Found in sentence {sentence_index + 1}, word position {word_index + 1}")
                        found= True
            if not found:
                print("WORD IS NOT FOUND!")


        # choice 3: Word Replacement
        # Feature 3: Replace all occurrences of a word and rebuild language models
        elif choice == '3':
            old_word = input("Word to replace: ").lower()
            new_word = input("New word: ").lower()

            if old_word in words_list:
                #Save a copy before editing
                undo_stack.append(words_list.copy())
                words_list = [new_word if word == old_word else word for word in words_list]
                clean_text= " ".join(words_list)
                sentences_list= clean_text.split(".")
                sentences_list= [s.strip() for s in sentences_list if s.strip()]

                # rebuild models after modification
                word_counts = collections.Counter(words_list)
                bigram_counts = build_bigram_counts(words_list)
                trigram_counts = build_trigram_counts(words_list)

                # rebuild Trie after modification so it keeps being updated
                trie= Trie()
                for word in words_list:
                  trie.insert(word)

                print(f"Successfully replaced all occurrences of '{old_word}' with '{new_word}'.")
            else:
                print(f"Word '{old_word}' not found.")

        # choice 4: Character Statistics
        # Feature 4: Compute character frequency excluding spaces
        elif choice == '4':
            full_text_no_spaces = "".join(words_list)
            char_counts = collections.Counter(full_text_no_spaces)
            print(f"\nTotal characters (excluding spaces): {len(full_text_no_spaces)}")
            print(f"Character occurrences: {dict(char_counts)}")

        # choice 5: Keyword Extraction
        # Feature 5: Extract top keywords by removing stop words and ranking by frequency
        elif choice == '5':
          print("\n--- Top 5 Keywords ---")
          from collections import deque, Counter

          # create a queue from the words list
          wordQueue= deque(words_list)
          filtered_counter= Counter()

          # process words sequentially using FIFO
          while wordQueue:
            currentWord= wordQueue.popleft()
            if currentWord not in stop_words:
              filtered_counter[currentWord] += 1

          # sort and get top 5
          sorted_keywords= sorted(filtered_counter.items(), key=lambda x: x[1], reverse=True)[:5]
          for word, freq in sorted_keywords:
            print(f"{word} -> {freq}")


        # choice 6: Sentiment Analysis
        # Feature 6: Perform simple sentence-level sentiment analysis using lexicon-based scoring
        elif choice == '6':
            sentence= input("Enter a sentence to analyze sentiment: ").lower()
            words= sentence.split()
            score= 0

            for i, word in enumerate(words):
                if word in positive_words:
                    if i > 0 and words[i-1] in negation_words:
                        score -= 1
                    else:
                        score += 1
                elif word in negative_words:
                    if i > 0 and words[i-1] in negation_words:
                        score += 1
                    else:
                        score -= 1

            if score > 0:
                print("Sentiment: Positive")
            elif score < 0:
                print("Sentiment: Negative")
            else:
                print("Sentiment: Neutral")

        # choice 7: Smart Autocomplete
        # Feature 7: Provide prefix-based word suggestions using Trie
        elif choice == '7':
            prefix = input("Enter a prefix: ").lower()
            suggestions = trie.search(prefix)

            if suggestions:
                print("Suggestions:", suggestions)
            else:
                print("No suggestions found.")

        # choice 8: Next Word Prediction
        # Feature 8: Predict next word using N-gram (Trigram/Bigram) frequency models
        elif choice == '8':
            user_input= input("Enter a word or phrase: ")
            suggestions= predict_next_word(user_input, bigram_counts, trigram_counts, top_n=5)

            if suggestions:
                print("Next word suggestions:", suggestions)
            else:
                print("No prediction available.")
        # choice 9: Undo Last Replacement using stack data structure
        elif choice == '9':
            if undo_stack:
                words_list = undo_stack.pop()

                # تحديث المتغيرات المعتمدة على words_list
                clean_text = " ".join(words_list)
                sentences_list = clean_text.split(".")
                sentences_list = [s.strip() for s in sentences_list if s.strip()]

                word_counts = collections.Counter(words_list)
                bigram_counts = build_bigram_counts(words_list)
                trigram_counts = build_trigram_counts(words_list)

                # rebuild Trie after undo so it keeps being updated as we did in the replacement choice
                trie= Trie()
                for word in words_list:
                  trie.insert(word)

                print("Last modification undone successfully.")
            else:
                print("Nothing to undo.")

        # choice 10: Exit
        elif choice == '10':
            print("Exiting Smart Text Analyzer...  See you again, Goodbye!")
            break

        else:
            print("Invalid choice. Please enter a number between 1 and 10.")

        input("\nPress Enter to return to the main menu...")

--- Welcome to The Intelligent Text Analyzer! ---
--- Please choose a Way to insert your text  ---
1. Direct Text Entry (d)
2. File Path Input (f)
Select (d/f): f
Please enter the full path to your text file: /content/mytext.txt

[Success] Loaded 25 words.

--- SMART TEXT ANALYZER MENU ---
1. Word Statistics (Total & Unique)
2. Search for a Word/Phrase
3. Word Replacement
4. Character Statistics
5. Keyword Extraction
6. Sentiment Analysis
7. Smart Autocomplete
8. Next Word Prediction
9. Undo Last Replacement
10. Exit Program
Select an option (1-10): 1

Total Number of Words: 25
Number of Unique Words: 23
Top 5 most frequent words: [('is', 3), ('artificial', 1), ('intelligence', 1), ('changing', 1), ('the', 1)]

Press Enter to return to the main menu...

--- SMART TEXT ANALYZER MENU ---
1. Word Statistics (Total & Unique)
2. Search for a Word/Phrase
3. Word Replacement
4. Character Statistics
5. Keyword Extraction
6. Sentiment Analysis
7. Smart Autocomplete
8. Next Word Prediction
9. Un